In [4]:
!pip install pandas boto3 scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 11.4 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [5]:
"""
Analyze NFL QB rushing performance in first playoff game vs. subsequent games.

Hypothesis: Do QBs rush more in their first playoff game?

Usage:
    python3 analysis/analyze_qb_first_playoff_rush.py
    
Or drag into notebook:
    from analysis.analyze_qb_first_playoff_rush import load_and_analyze
    df = load_and_analyze()
"""

import pandas as pd
import boto3
from io import StringIO

S3_BUCKET = 'nfl-betting-mt'
S3_PREFIX = 'data/01_input/espn_web/playoffs/qb/gamelogs'


def load_all_qb_files():
    """Load all QB playoff files from S3 and mark first playoff game."""
    s3 = boto3.client('s3')
    
    # List all QB files
    response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_PREFIX)
    files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.csv') and 'all_qb' not in obj['Key']]
    
    dfs = []
    for file_key in files:
        # Read CSV from S3
        obj = s3.get_object(Bucket=S3_BUCKET, Key=file_key)
        df = pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')))
        
        # Sort by season and date
        df = df.sort_values(['season', 'date'])
        
        # Mark first playoff game
        df['first_playoff_game_binary'] = False
        df.iloc[0, df.columns.get_loc('first_playoff_game_binary')] = True
        
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)


def analyze_first_game_rushing(df):
    """Compare rushing yards: first playoff game vs. all others."""
    
    # Convert to numeric
    df['rushing_yds'] = pd.to_numeric(df['rushing_yds'], errors='coerce')
    
    # Split into first vs. other games
    first_games = df[df['first_playoff_game_binary'] == True]
    other_games = df[df['first_playoff_game_binary'] == False]
    
    print("\n" + "="*80)
    print("QB RUSHING: FIRST PLAYOFF GAME vs. OTHER GAMES")
    print("="*80)
    
    print(f"\nFirst Playoff Games: {len(first_games)} games")
    print(f"  Mean rushing yards: {first_games['rushing_yds'].mean():.1f}")
    print(f"  Median rushing yards: {first_games['rushing_yds'].median():.1f}")
    print(f"  Std dev: {first_games['rushing_yds'].std():.1f}")
    
    print(f"\nOther Playoff Games: {len(other_games)} games")
    print(f"  Mean rushing yards: {other_games['rushing_yds'].mean():.1f}")
    print(f"  Median rushing yards: {other_games['rushing_yds'].median():.1f}")
    print(f"  Std dev: {other_games['rushing_yds'].std():.1f}")
    
    diff = first_games['rushing_yds'].mean() - other_games['rushing_yds'].mean()
    print(f"\nDifference: {diff:+.1f} yards (first game vs. others)")
    
    # Statistical test
    from scipy import stats
    t_stat, p_value = stats.ttest_ind(first_games['rushing_yds'].dropna(), 
                                       other_games['rushing_yds'].dropna())
    print(f"T-test p-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print("✅ Statistically significant difference!")
    else:
        print("❌ No significant difference")
    
    print("="*80)
    
    return df


def load_and_analyze():
    """Main function: load data and run analysis."""
    print("Loading QB playoff data from S3...")
    df = load_all_qb_files()
    print(f"✅ Loaded {len(df)} total playoff games")
    
    df = analyze_first_game_rushing(df)
    
    return df


if __name__ == '__main__':
    df = load_and_analyze()
    
    # Save for further analysis
    # df.to_csv('data/04_output/qb_playoff_rush_analysis.csv', index=False)
    # print(f"\n💾 Saved to: data/04_output/qb_playoff_rush_analysis.csv")



Loading QB playoff data from S3...
✅ Loaded 584 total playoff games

QB RUSHING: FIRST PLAYOFF GAME vs. OTHER GAMES

First Playoff Games: 109 games
  Mean rushing yards: 16.7
  Median rushing yards: 9.0
  Std dev: 24.4

Other Playoff Games: 475 games
  Mean rushing yards: 14.4
  Median rushing yards: 5.0
  Std dev: 22.6

Difference: +2.3 yards (first game vs. others)
T-test p-value: 0.3444
❌ No significant difference


In [6]:
df

,date,opponent,result,completions,attempts,passing_yards,passing_cmp%,passing_avg,passing_tds,interceptions,...,rushing_yds,rushing_avg,rushing_td,rushing_lng,season,athlete,athlete_id,playoff_round,passing_qbr,first_playoff_game_binary
0,Sun 1/10,@ARI,L51-45 OT,28,42.0,423,66.7,10.1,4,1,...,13.0,4.3,1.0,13.0,2009,Aaron Rodgers,8439,Wild Card,87.8,True
1,Sat 1/15,@ATL,W48-21,31,36.0,366,86.1,10.2,3,0,...,13.0,6.5,1.0,7.0,2010,Aaron Rodgers,8439,Divisional,95.9,False
2,Sun 1/23,@CHI,W21-14,17,30.0,244,56.7,8.1,0,2,...,39.0,5.6,1.0,25.0,2010,Aaron Rodgers,8439,Conference Championship,68.3,False
3,Sun 1/9,@PHI,W21-16,18,27.0,180,66.7,6.7,3,0,...,4.0,1.3,0.0,8.0,2010,Aaron Rodgers,8439,Wild Card,82.2,False
4,Sun 2/6,vsPIT,W31-25,24,39.0,304,61.5,7.8,3,0,...,-2.0,-1.0,0.0,-1.0,2010,Aaron Rodgers,8439,Super Bowl,83.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
579,Sun 1/15,@CIN,L24-17,17,29.0,226,58.6,7.8,2,1,...,54.0,6.0,0.0,35.0,2022,Tyler Huntley,4035671,Wild Card,75.0,False
580,Sun 1/31,vsIRV*,L49-27,8,14.0,120,57.1,8.6,1,3,...,15.0,7.5,0.0,11.0,2015,Tyrod Taylor,14163,Conference Championship,-,True
581,Sun 1/7,@JAX,L10-3,17,37.0,134,45.9,3.6,0,1,...,27.0,3.9,0.0,9.0,2017,Tyrod Taylor,14163,Wild Card,48.7,False
582,Sun 1/6,@SD,L17-6,16,29.0,138,55.2,4.8,0,1,...,12.0,6.0,0.0,9.0,2007,Vince Young,9589,Wild Card,51.3,True


In [11]:
# get rushing props data

# ...

"""
Load all NFL QB playoff data and rushing props

Data sources:
1. QB game logs: s3://nfl-betting-mt/data/01_input/espn_web/playoffs/qb/gamelogs/
2. Rushing props: s3://the-odds-api-mt/nfl/historical_player_props/{season}/player_rush_yds/

Context:
- QB data: 584 playoff games, 109 QBs, 2001-2024
- Props data: 2023-24, 2024-25, 2025-26 playoff seasons
"""

import pandas as pd
import boto3
from io import StringIO
import os

# =============================================================================
# CONFIGURATION
# =============================================================================

# S3 buckets
NFL_BETTING_BUCKET = 'nfl-betting-mt'
ODDS_API_BUCKET = 'the-odds-api-mt'

# S3 paths
QB_GAMELOGS_PREFIX = 'data/01_input/espn_web/playoffs/qb/gamelogs'
PROPS_PREFIX = 'nfl/historical_player_props'

# Seasons with playoff props data
PROP_SEASONS = ['2023-24', '2024-25', '2025-26']

s3_client = boto3.client('s3')

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def list_s3_files(bucket, prefix):
    """List all files in S3 bucket with prefix"""
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    files = []
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith('.csv'):
                    files.append(obj['Key'])
    
    return files

def load_csv_from_s3(bucket, key):
    """Load CSV file from S3 into DataFrame"""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')))

# =============================================================================
# LOAD QB GAME LOGS
# =============================================================================

def load_all_qb_gamelogs():
    """Load master QB playoff game log file"""
    print("Loading QB playoff game logs from S3...")
    
    # List files in QB gamelogs directory
    files = list_s3_files(NFL_BETTING_BUCKET, QB_GAMELOGS_PREFIX)
    
    # Find master file (most recent)
    master_files = [f for f in files if 'nfl_all_qb_playoff_gamelogs' in f]
    
    if not master_files:
        raise ValueError("No master QB gamelog file found!")
    
    # Use most recent (sorted by date in filename)
    master_file = sorted(master_files)[-1]
    
    print(f"  Loading: {master_file}")
    df = load_csv_from_s3(NFL_BETTING_BUCKET, master_file)
    
    print(f"  ✅ Loaded {len(df)} playoff games for {df['athlete'].nunique()} QBs")
    return df

# =============================================================================
# LOAD RUSHING PROPS
# =============================================================================

def load_rushing_props_for_season(season):
    """Load all rushing props for a specific season"""
    print(f"\n  {season}:")
    
    season_prefix = f"{PROPS_PREFIX}/{season}/player_rush_yds"
    files = list_s3_files(ODDS_API_BUCKET, season_prefix)
    
    if not files:
        print(f"    ⚠️  No files found")
        return pd.DataFrame()
    
    print(f"    Found {len(files)} files")
    
    # Load all CSV files for this season
    dfs = []
    for file_key in files:
        df = load_csv_from_s3(ODDS_API_BUCKET, file_key)
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"    ✅ Loaded {len(combined)} prop records")
    
    return combined

def load_all_rushing_props():
    """Load rushing props for all available seasons"""
    print("\nLoading historical rushing props from S3...")
    
    all_props = []
    
    for season in PROP_SEASONS:
        df = load_rushing_props_for_season(season)
        if len(df) > 0:
            all_props.append(df)
    
    if not all_props:
        print("\n  ⚠️  No props data found")
        return pd.DataFrame()
    
    combined = pd.concat(all_props, ignore_index=True)
    
    print(f"\n  ✅ Total: {len(combined)} prop records across {len(all_props)} seasons")
    return combined

# =============================================================================
# MAIN LOAD FUNCTION
# =============================================================================

def load_and_analyze():
    """Main function: load data and run analysis"""
    print("="*80)
    print("LOADING QB PLAYOFF DATA")
    print("="*80)
    
    # Load QB game logs
    df_qb = load_all_qb_gamelogs()
    
    # Load rushing props
    df_props = load_all_rushing_props()
    
    print(f"\n{'='*80}")
    print("DATA LOADED")
    print(f"{'='*80}")
    print(f"QB game logs: {len(df_qb)} rows")
    print(f"Rushing props: {len(df_props)} rows")
    print(f"\nQB columns: {list(df_qb.columns)}")
    if len(df_props) > 0:
        print(f"Props columns: {list(df_props.columns)}")
    
    return df_qb, df_props

# =============================================================================
# EXECUTION
# =============================================================================

# Load all data
df_qb, df_props = load_and_analyze()

# Show samples
print(f"\n{'='*80}")
print("SAMPLE QB DATA")
print(f"{'='*80}")
print(df_qb.head())

if len(df_props) > 0:
    print(f"\n{'='*80}")
    print("SAMPLE PROPS DATA")
    print(f"{'='*80}")
    print(df_props.head())

LOADING QB PLAYOFF DATA
Loading QB playoff game logs from S3...
  Loading: data/01_input/espn_web/playoffs/qb/gamelogs/nfl_all_qb_playoff_gamelogs_2000_2024_20260112.csv
  ✅ Loaded 584 playoff games for 109 QBs

Loading historical rushing props from S3...

  2023-24:
    Found 7 files
    ✅ Loaded 582 prop records

  2024-25:
    Found 7 files
    ✅ Loaded 541 prop records

  2025-26:
    Found 3 files
    ✅ Loaded 256 prop records

  ✅ Total: 1379 prop records across 3 seasons

DATA LOADED
QB game logs: 584 rows
Rushing props: 1379 rows

QB columns: ['date', 'opponent', 'result', 'completions', 'attempts', 'passing_yards', 'passing_cmp%', 'passing_avg', 'passing_tds', 'interceptions', 'passing_lng', 'passing_sack', 'passing_rtg', 'rushing_car', 'rushing_yds', 'rushing_avg', 'rushing_td', 'rushing_lng', 'season', 'athlete', 'athlete_id', 'playoff_round', 'passing_qbr']
Props columns: ['player', 'away_team', 'home_team', 'game_time', 'market', 'prop_line', 'over_odds', 'under_odds', 'bo

In [10]:
# join

In [12]:
"""
Load and Join NFL QB Playoff Game Logs with Rushing Props

Context:
--------
Combines two data sources:
1. QB playoff game logs (2001-2024): ESPN box scores with actual rushing yards
2. Historical rushing props (2023-26): The Odds API betting lines

Goal:
-----
Join these datasets to analyze:
- How QBs perform vs their rushing props in playoff games
- First playoff game rushing props specifically
- Bookmaker line accuracy

Data Sources:
-------------
1. s3://nfl-betting-mt/data/01_input/espn_web/playoffs/qb/gamelogs/
   - 584 playoff games, 109 QBs, seasons 2001-2024
   - Columns: athlete, season, date, opponent, attempts, passing_yards, rushing_yds, etc.

2. s3://the-odds-api-mt/nfl/historical_player_props/{season}/player_rush_yds/
   - Betting lines from 2023-24, 2024-25, 2025-26 playoff seasons
   - Columns: player_name, commence_time, bookmaker, line, over_price, under_price, etc.

Join Strategy:
--------------
Match on:
- Player name (fuzzy matching needed: "Patrick Mahomes" vs "P. Mahomes")
- Date (standardize both to YYYY-MM-DD)
- Season (ensure props match game season)

Usage:
------
cd betting
python analysis/load_qb_playoff_rushing_data.py

Returns:
--------
- df_qb: All QB playoff game logs
- df_props: All rushing props
- df_joined: Merged dataset (QB games with props)

Created: 2026-01-12
Author: Thomas Myles
"""

import pandas as pd
import boto3
from io import StringIO
import os
from datetime import datetime
import re

# =============================================================================
# CONFIGURATION
# =============================================================================

# S3 buckets
NFL_BETTING_BUCKET = 'nfl-betting-mt'
ODDS_API_BUCKET = 'the-odds-api-mt'

# S3 paths
QB_GAMELOGS_PREFIX = 'data/01_input/espn_web/playoffs/qb/gamelogs'
PROPS_PREFIX = 'nfl/historical_player_props'

# Seasons with playoff props data
PROP_SEASONS = ['2023-24', '2024-25', '2025-26']

s3_client = boto3.client('s3')

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def list_s3_files(bucket, prefix):
    """List all files in S3 bucket with prefix"""
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
    
    files = []
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                if obj['Key'].endswith('.csv'):
                    files.append(obj['Key'])
    
    return files

def load_csv_from_s3(bucket, key):
    """Load CSV file from S3 into DataFrame"""
    obj = s3_client.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(StringIO(obj['Body'].read().decode('utf-8')))

def normalize_name(name):
    """
    Normalize player name for matching.
    
    Examples:
        "Patrick Mahomes II" -> "patrick mahomes"
        "P. Mahomes" -> "p mahomes"
        "Jalen Hurts" -> "jalen hurts"
    """
    name = str(name).lower().strip()
    # Remove suffixes (Jr, Sr, II, III, IV)
    name = re.sub(r'\s+(jr\.?|sr\.?|ii|iii|iv)$', '', name, flags=re.IGNORECASE)
    # Remove periods and extra spaces
    name = re.sub(r'\.', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name

def parse_espn_date(date_str, season):
    """
    Parse ESPN date format to YYYY-MM-DD.
    
    ESPN format: "Sat 1/11" or "Sun 2/9"
    Need to infer year from season (playoffs span two calendar years)
    
    Args:
        date_str: Like "Sat 1/11" or "Sun 2/9"
        season: Like 2024 (means 2024-25 season)
    
    Returns:
        str: YYYY-MM-DD format
    """
    # Extract month and day
    match = re.search(r'(\d{1,2})/(\d{1,2})', date_str)
    if not match:
        return None
    
    month, day = int(match.group(1)), int(match.group(2))
    
    # Playoffs are Jan/Feb of NEXT calendar year
    # Example: 2024 season = playoffs in Jan/Feb 2025
    year = season + 1
    
    # Handle edge case: if month is 12, it's pre-playoffs (shouldn't happen)
    if month == 12:
        year = season
    
    return f"{year}-{month:02d}-{day:02d}"

def parse_odds_api_date(date_str):
    """
    Parse Odds API datetime to YYYY-MM-DD.
    
    Odds API format: "2025-01-11T18:30:00Z" or similar ISO format
    
    Returns:
        str: YYYY-MM-DD format
    """
    try:
        dt = pd.to_datetime(date_str)
        return dt.strftime('%Y-%m-%d')
    except:
        return None

# =============================================================================
# LOAD QB GAME LOGS
# =============================================================================

def load_all_qb_gamelogs():
    """Load master QB playoff game log file"""
    print("Loading QB playoff game logs from S3...")
    
    # List files in QB gamelogs directory
    files = list_s3_files(NFL_BETTING_BUCKET, QB_GAMELOGS_PREFIX)
    
    # Find master file (most recent)
    master_files = [f for f in files if 'nfl_all_qb_playoff_gamelogs' in f]
    
    if not master_files:
        raise ValueError("No master QB gamelog file found!")
    
    # Find the file matching our prop seasons (2023-2025)
    # Prefer recent narrow range over full history, as we only have props for 2023-26
    def parse_file_range(filename):
        """Extract start_year, end_year, date from filename"""
        # Example: nfl_all_qb_playoff_gamelogs_2023_2025_20260112.csv
        import re
        match = re.search(r'(\d{4})_(\d{4})_(\d{8})', filename)
        if match:
            start_year = int(match.group(1))
            end_year = int(match.group(2))
            date = int(match.group(3))
            
            # Prioritize files that cover our prop seasons (2023-2025)
            covers_prop_seasons = (start_year <= 2023 and end_year >= 2025)
            
            # Sort by: covers prop seasons (True first), then most recent date
            return (covers_prop_seasons, date)
        return (False, 0)
    
    # Sort to get file that covers prop seasons with most recent date
    master_file = sorted(master_files, key=parse_file_range, reverse=True)[0]
    
    print(f"  Loading: {master_file}")
    df = load_csv_from_s3(NFL_BETTING_BUCKET, master_file)
    
    # Clean and standardize
    df['athlete_normalized'] = df['athlete'].apply(normalize_name)
    df['game_date'] = df.apply(lambda row: parse_espn_date(row['date'], row['season']), axis=1)
    
    # Add season_str for matching with props (e.g., 2024 -> "2024-25")
    df['season_str'] = df['season'].apply(lambda x: f"{x}-{str(x+1)[-2:]}")
    
    # Add flag for FIRST playoff game for each QB
    # Sort by athlete and date, then mark first game
    df = df.sort_values(['athlete', 'game_date']).reset_index(drop=True)
    df['is_first_playoff_game'] = ~df.duplicated(subset=['athlete'], keep='first')
    
    # Count and show first playoff games
    first_games_count = df['is_first_playoff_game'].sum()
    
    print(f"  ✅ Loaded {len(df)} playoff games for {df['athlete'].nunique()} QBs")
    print(f"     {first_games_count} first playoff games identified")
    
    return df

# =============================================================================
# LOAD RUSHING PROPS
# =============================================================================

def load_rushing_props_for_season(season):
    """Load all rushing props for a specific season"""
    print(f"\n  {season}:")
    
    season_prefix = f"{PROPS_PREFIX}/{season}/player_rush_yds"
    files = list_s3_files(ODDS_API_BUCKET, season_prefix)
    
    if not files:
        print(f"    ⚠️  No files found")
        return pd.DataFrame()
    
    print(f"    Found {len(files)} files")
    
    # Load all CSV files for this season
    dfs = []
    for file_key in files:
        df = load_csv_from_s3(ODDS_API_BUCKET, file_key)
        dfs.append(df)
    
    combined = pd.concat(dfs, ignore_index=True)
    print(f"    ✅ Loaded {len(combined)} prop records")
    
    return combined

def load_all_rushing_props():
    """Load rushing props for all available seasons"""
    print("\nLoading historical rushing props from S3...")
    
    all_props = []
    
    for season in PROP_SEASONS:
        df = load_rushing_props_for_season(season)
        if len(df) > 0:
            df['season_str'] = season
            all_props.append(df)
    
    if not all_props:
        print("\n  ⚠️  No props data found")
        return pd.DataFrame()
    
    combined = pd.concat(all_props, ignore_index=True)
    
    # Clean and standardize
    # Props data has player name in 'player' column
    if 'player' not in combined.columns:
        raise ValueError(f"Cannot find 'player' column in props data. Columns: {combined.columns.tolist()}")
    
    combined['player_normalized'] = combined['player'].apply(normalize_name)
    
    # Parse game_time to game_date
    if 'game_time' in combined.columns:
        combined['game_date'] = combined['game_time'].apply(parse_odds_api_date)
    else:
        raise ValueError("Cannot find game_time column in props data")
    
    print(f"\n  ✅ Total: {len(combined)} prop records across {len(all_props)} seasons")
    return combined

# =============================================================================
# JOIN DATASETS
# =============================================================================

def join_qb_games_with_props(df_qb, df_props):
    """
    Join QB game logs with rushing props.
    
    Join Strategy (by player + game):
    ---------------------------------
    For each QB's playoff game, find the betting prop for THAT specific game:
    
    1. Player match: "Patrick Mahomes" (QB data) = "Patrick Mahomes" (prop data)
    2. Game match: "2025-01-18" (game date) = "2025-01-18" (prop commence date)
    3. Season match: "2024-25" (ensures correct season)
    
    Example:
        QB Game: Mahomes, 2025-01-18, rushed for 45 yards
        Prop: Mahomes, 2025-01-18, line was 25.5 yards
        Result: Joined row shows actual (45) vs line (25.5)
    
    Returns:
        DataFrame: Each row = one QB's game with actual rushing yards + prop line
    """
    print("\n" + "="*80)
    print("JOINING QB GAMES WITH PROPS (BY PLAYER + GAME)")
    print("="*80)
    
    if len(df_props) == 0:
        print("⚠️  No props data to join")
        return pd.DataFrame()
    
    # Show BEFORE filtering: all QB data
    print(f"\nBEFORE FILTERING (All QB data):")
    print(f"  Total QB playoff games: {len(df_qb)}")
    print(f"  QBs: {df_qb['athlete'].nunique()}")
    print(f"  Seasons: {df_qb['season'].min()}-{df_qb['season'].max()}")
    
    # Filter QB data to only prop seasons (2023-24 and later)
    df_qb_prop_seasons = df_qb[df_qb['season_str'].isin(PROP_SEASONS)].copy()
    
    print(f"\nAFTER FILTERING (Only prop seasons: {', '.join(PROP_SEASONS)}):")
    print(f"  QB games in prop seasons: {len(df_qb_prop_seasons)}")
    print(f"  QBs in prop seasons: {df_qb_prop_seasons['athlete'].nunique()}")
    print(f"  Unique game dates: {df_qb_prop_seasons['game_date'].nunique()}")
    
    # Inner join: only keep QB games that have matching props
    # Join keys ensure we match EACH PLAYER'S SPECIFIC GAME
    df_joined = df_qb_prop_seasons.merge(
        df_props,
        left_on=['athlete_normalized', 'game_date', 'season_str'],
        right_on=['player_normalized', 'game_date', 'season_str'],
        how='inner',
        suffixes=('_actual', '_prop')  # Distinguish overlapping columns
    )
    
    print(f"\nJOIN RESULTS:")
    print(f"  ✅ Matched: {len(df_joined)} QB game + prop combinations")
    print(f"     Unique QBs with props: {df_joined['athlete'].nunique()}")
    print(f"     Unique games with props: {df_joined['game_date'].nunique()}")
    
    # Calculate join rate
    join_rate_games = (len(df_joined) / len(df_qb_prop_seasons) * 100) if len(df_qb_prop_seasons) > 0 else 0
    join_rate_qbs = (df_joined['athlete'].nunique() / df_qb_prop_seasons['athlete'].nunique() * 100) if df_qb_prop_seasons['athlete'].nunique() > 0 else 0
    
    print(f"\n  📊 JOIN RATE:")
    print(f"     Games: {join_rate_games:.1f}% ({len(df_joined)}/{len(df_qb_prop_seasons)})")
    print(f"     QBs: {join_rate_qbs:.1f}% ({df_joined['athlete'].nunique()}/{df_qb_prop_seasons['athlete'].nunique()})")
    
    # Sample matched games
    if len(df_joined) > 0:
        print(f"\n  Sample Matched Games (multiple bookmakers per game):")
        for _, row in df_joined[['athlete', 'game_date', 'opponent']].drop_duplicates().head(5).iterrows():
            print(f"    - {row['athlete']} on {row['game_date']} vs {row['opponent']}")
    
    return df_joined

def get_consensus_lines(df_joined):
    """
    Aggregate multiple bookmaker lines to get consensus (median) for each player/game.
    
    Strategy:
    ---------
    For each unique player + game combination:
    1. We have 5-10 different bookmaker lines (DraftKings, FanDuel, etc.)
    2. Take MEDIAN of prop_line across all bookmakers
    3. Result: ONE row per player per game with consensus line
    
    Example:
        Before:
            Mahomes, 2025-01-18, DraftKings, line=25.5
            Mahomes, 2025-01-18, FanDuel, line=26.5
            Mahomes, 2025-01-18, BetMGM, line=25.0
        
        After:
            Mahomes, 2025-01-18, consensus_line=25.5 (median)
    
    Returns:
        DataFrame: One row per player per game with consensus line
    """
    print("\n" + "="*80)
    print("CALCULATING CONSENSUS LINES (MEDIAN BY PLAYER/GAME)")
    print("="*80)
    
    if len(df_joined) == 0:
        print("⚠️  No joined data to aggregate")
        return pd.DataFrame()
    
    print(f"\nBefore aggregation: {len(df_joined)} rows (multiple bookmakers)")
    
    # Determine column names after merge (they might have _actual or _prop suffixes)
    season_col = 'season_actual' if 'season_actual' in df_joined.columns else 'season'
    
    # Group by player + game and aggregate
    groupby_cols = ['athlete', 'athlete_id', 'game_date', season_col, 'season_str', 'opponent']
    
    # Add playoff_round if it exists
    if 'playoff_round' in df_joined.columns:
        groupby_cols.append('playoff_round')
    
    # Build aggregation dict
    agg_dict = {
        # Actual game stats (same across all bookmakers, just take first)
        'rushing_yds': 'first',
        'rushing_car': 'first',
        'rushing_avg': 'first',
        'passing_yards': 'first',
        'passing_tds': 'first',
        'attempts': 'first',
        'result': 'first',
        
        # First playoff game flag
        'is_first_playoff_game': 'first',
        
        # Consensus prop line (MEDIAN across bookmakers)
        'prop_line': 'median',
        
        # Count how many bookmakers
        'bookmaker': 'count',
    }
    
    consensus = df_joined.groupby(
        groupby_cols,
        dropna=False
    ).agg(agg_dict).reset_index()
    
    # Rename season column back to 'season' if it was suffixed
    if season_col == 'season_actual':
        consensus = consensus.rename(columns={'season_actual': 'season'})
    
    # Rename for clarity
    consensus = consensus.rename(columns={
        'prop_line': 'consensus_line',
        'bookmaker': 'num_bookmakers'
    })
    
    # Calculate difference: actual vs consensus
    consensus['diff_vs_consensus'] = consensus['rushing_yds'] - consensus['consensus_line']
    consensus['beat_line'] = consensus['diff_vs_consensus'] > 0
    
    print(f"After aggregation: {len(consensus)} rows (one per player/game)")
    print(f"\nConsensus Stats:")
    print(f"  Unique QBs: {consensus['athlete'].nunique()}")
    print(f"  Unique games: {consensus['game_date'].nunique()}")
    print(f"  Avg bookmakers per game: {consensus['num_bookmakers'].mean():.1f}")
    
    # Show sample
    print(f"\nSample Consensus Lines:")
    display_cols = ['athlete', 'game_date', 'is_first_playoff_game', 'rushing_yds', 'consensus_line', 'diff_vs_consensus', 'num_bookmakers']
    print(consensus[display_cols].head(10).to_string(index=False))
    
    # Show first playoff games specifically
    first_games = consensus[consensus['is_first_playoff_game'] == True].sort_values('game_date')
    print(f"\n🏈 FIRST PLAYOFF GAMES:")
    print(f"   {len(first_games)} QBs in their first playoff game")
    if len(first_games) > 0:
        print(f"\n   All First Playoff Games:")
        print(f"     {'HIT':<4} {'QB':<20} {'Date':<12} {'Opponent':<8} {'Result':<8} | {'Rush Yds':>8} vs {'Line':>5} | {'Diff':>6} | {'Pass Yds':<8} {'Pass TD':<7}")
        print(f"     {'-'*4} {'-'*20} {'-'*12} {'-'*8} {'-'*8}   {'-'*8}    {'-'*5}   {'-'*6}   {'-'*8} {'-'*7}")
        for _, row in first_games.iterrows():
            result_emoji = "✅" if row['beat_line'] else "❌"
            diff = row['diff_vs_consensus']
            result_str = row.get('result', 'N/A')
            opponent = row.get('opponent', 'N/A')
            pass_yds = row.get('passing_yards', 0)
            pass_tds = row.get('passing_tds', 0)
            
            print(f"     {result_emoji:<4} {row['athlete']:20s} {row['game_date']:<12} {opponent:<8} {result_str:<8} | {row['rushing_yds']:>8.0f} vs {row['consensus_line']:>5.1f} | {diff:>+6.1f} | {pass_yds:>8.0f} {pass_tds:>7.0f}")
    
    # === GROUP BY FIRST PLAYOFF GAME: OVER/UNDER RATES ===
    print(f"\n" + "="*80)
    print("OVER/UNDER RATES BY FIRST PLAYOFF GAME STATUS")
    print("="*80)
    
    groupby_first_game = consensus.groupby('is_first_playoff_game').agg({
        'beat_line': ['count', 'sum', 'mean'],
        'diff_vs_consensus': 'mean'
    }).round(3)
    
    # Flatten column names
    groupby_first_game.columns = ['total_games', 'beat_line_count', 'beat_line_rate', 'avg_diff']
    groupby_first_game = groupby_first_game.reset_index()
    
    # Add formatted columns for display
    groupby_first_game['beat_line_pct'] = (groupby_first_game['beat_line_rate'] * 100).round(1)
    groupby_first_game['record'] = groupby_first_game.apply(
        lambda x: f"{int(x['beat_line_count'])}/{int(x['total_games'])}", axis=1
    )
    
    # Display results
    for _, row in groupby_first_game.iterrows():
        first_game_status = "FIRST PLAYOFF GAME" if row['is_first_playoff_game'] else "VETERAN (not first)"
        emoji = "🆕" if row['is_first_playoff_game'] else "🏆"
        
        print(f"\n{emoji} {first_game_status}:")
        print(f"   Total Games: {int(row['total_games'])}")
        print(f"   Beat Line: {row['record']} ({row['beat_line_pct']:.1f}%)")
        print(f"   Avg Difference: {row['avg_diff']:+.1f} yards vs consensus")
    
    # Calculate the edge
    first_rate = groupby_first_game[groupby_first_game['is_first_playoff_game'] == True]['beat_line_pct'].values
    veteran_rate = groupby_first_game[groupby_first_game['is_first_playoff_game'] == False]['beat_line_pct'].values
    
    if len(first_rate) > 0 and len(veteran_rate) > 0:
        edge = first_rate[0] - veteran_rate[0]
        print(f"\n{'🔥' if edge > 0 else '❄️'} EDGE: First playoff game QBs hit at {edge:+.1f}% higher rate")
        
        if edge > 5:
            print(f"   💡 INSIGHT: First playoff game QBs significantly OUTPERFORM their lines!")
        elif edge < -5:
            print(f"   ⚠️  INSIGHT: First playoff game QBs significantly UNDERPERFORM their lines!")
        else:
            print(f"   ℹ️  INSIGHT: No significant edge detected (within 5%)")
    
    # === DETAILED BREAKDOWN: ALL QB/GAME COMBINATIONS ===
    print(f"\n" + "="*80)
    print("DETAILED BREAKDOWN: ALL QB/GAME COMBINATIONS")
    print("="*80)
    
    # First playoff games
    veteran_games = consensus[consensus['is_first_playoff_game'] == False].sort_values('game_date')
    
    print(f"\n🆕 FIRST PLAYOFF GAMES ({len(first_games)} games):")
    print(f"{'HIT':<4} {'QB':<20} {'Date':<12} {'Opponent':<8} {'Result':<8} | {'Rush Yds':>8} vs {'Line':>5} | {'Diff':>6}")
    print(f"{'-'*4} {'-'*20} {'-'*12} {'-'*8} {'-'*8}   {'-'*8}    {'-'*5}   {'-'*6}")
    for _, row in first_games.iterrows():
        result_emoji = "✅" if row['beat_line'] else "❌"
        diff = row['diff_vs_consensus']
        result_str = row.get('result', 'N/A')
        opponent = row.get('opponent', 'N/A')
        print(f"{result_emoji:<4} {row['athlete']:20s} {row['game_date']:<12} {opponent:<8} {result_str:<8} | {row['rushing_yds']:>8.0f} vs {row['consensus_line']:>5.1f} | {diff:>+6.1f}")
    
    print(f"\n🏆 VETERAN QB GAMES ({len(veteran_games)} games):")
    print(f"{'HIT':<4} {'QB':<20} {'Date':<12} {'Opponent':<8} {'Result':<8} | {'Rush Yds':>8} vs {'Line':>5} | {'Diff':>6}")
    print(f"{'-'*4} {'-'*20} {'-'*12} {'-'*8} {'-'*8}   {'-'*8}    {'-'*5}   {'-'*6}")
    for _, row in veteran_games.iterrows():
        result_emoji = "✅" if row['beat_line'] else "❌"
        diff = row['diff_vs_consensus']
        result_str = row.get('result', 'N/A')
        opponent = row.get('opponent', 'N/A')
        print(f"{result_emoji:<4} {row['athlete']:20s} {row['game_date']:<12} {opponent:<8} {result_str:<8} | {row['rushing_yds']:>8.0f} vs {row['consensus_line']:>5.1f} | {diff:>+6.1f}")
    
    return consensus

# =============================================================================
# MAIN FUNCTION
# =============================================================================

def main():
    """Main function: load and join all data"""
    print("="*80)
    print("NFL QB PLAYOFF RUSHING ANALYSIS")
    print("="*80)
    
    # Step 1: Load QB game logs
    df_qb = load_all_qb_gamelogs()
    
    # Step 2: Load rushing props
    df_props = load_all_rushing_props()
    
    # Step 3: Join datasets (multiple bookmakers per game)
    df_joined = join_qb_games_with_props(df_qb, df_props)
    
    # Step 4: Get consensus lines (one row per player/game)
    df_consensus = get_consensus_lines(df_joined)
    
    # Step 5: Summary
    print(f"\n{'='*80}")
    print("FINAL DATA SUMMARY")
    print(f"{'='*80}")
    print(f"QB game logs: {len(df_qb)} games (all history)")
    print(f"Rushing props: {len(df_props)} prop lines (all bookmakers)")
    print(f"Joined (raw): {len(df_joined)} rows (multiple bookmakers per game)")
    print(f"Consensus: {len(df_consensus)} rows (one per player/game)")
    
    if len(df_consensus) > 0:
        print(f"\nConsensus Performance:")
        beat_pct = (df_consensus['beat_line'].sum() / len(df_consensus) * 100)
        print(f"  QBs that beat consensus: {df_consensus['beat_line'].sum()}/{len(df_consensus)} ({beat_pct:.1f}%)")
        print(f"  Avg difference: {df_consensus['diff_vs_consensus'].mean():.1f} yards")
    
    return df_qb, df_props, df_joined, df_consensus

# =============================================================================
# EXECUTION
# =============================================================================

if __name__ == "__main__":
    df_qb, df_props, df_joined, df_consensus = main()



NFL QB PLAYOFF RUSHING ANALYSIS
Loading QB playoff game logs from S3...
  Loading: data/01_input/espn_web/playoffs/qb/gamelogs/nfl_all_qb_playoff_gamelogs_2023_2025_20260112.csv
  ✅ Loaded 63 playoff games for 23 QBs
     23 first playoff games identified

Loading historical rushing props from S3...

  2023-24:
    Found 7 files
    ✅ Loaded 582 prop records

  2024-25:
    Found 7 files
    ✅ Loaded 541 prop records

  2025-26:
    Found 3 files
    ✅ Loaded 256 prop records

  ✅ Total: 1379 prop records across 3 seasons

JOINING QB GAMES WITH PROPS (BY PLAYER + GAME)

BEFORE FILTERING (All QB data):
  Total QB playoff games: 63
  QBs: 23
  Seasons: 2023-2025

AFTER FILTERING (Only prop seasons: 2023-24, 2024-25, 2025-26):
  QB games in prop seasons: 63
  QBs in prop seasons: 23
  Unique game dates: 17

JOIN RESULTS:
  ✅ Matched: 294 QB game + prop combinations
     Unique QBs with props: 18
     Unique games with props: 15

  📊 JOIN RATE:
     Games: 466.7% (294/63)
     QBs: 78.3% (

In [13]:
df_qb

,date,opponent,result,completions,attempts,passing_yards,passing_cmp%,passing_avg,passing_tds,interceptions,...,rushing_td,rushing_lng,season,athlete,athlete_id,playoff_round,athlete_normalized,game_date,season_str,is_first_playoff_game
0,Mon 1/15,vsPHI,W32-9,22,36,337,61.1,9.4,3,0,...,0.0,9.0,2023,Baker Mayfield,3052587,Wild Card,baker mayfield,2024-01-15,2023-24,True
1,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,0.0,14.0,2023,Baker Mayfield,3052587,Divisional,baker mayfield,2024-01-21,2023-24,False
2,Sun 1/12,vsWSH,L23-20,15,18,185,83.3,10.3,2,0,...,0.0,18.0,2024,Baker Mayfield,3052587,Wild Card,baker mayfield,2025-01-12,2024-25,False
3,Sun 1/12,@BUF,L31-7,13,22,144,59.1,6.5,1,0,...,0.0,18.0,2024,Bo Nix,4426338,Wild Card,bo nix,2025-01-12,2024-25,True
4,Sat 1/20,vsGB,W24-21,23,39,252,59.0,6.5,1,0,...,0.0,9.0,2023,Brock Purdy,4361741,Divisional,brock purdy,2024-01-20,2023-24,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,Sun 2/9,vsPHI,L40-22,21,32,257,65.6,8.0,3,2,...,0.0,8.0,2024,Patrick Mahomes,3139477,Super Bowl,patrick mahomes,2025-02-09,2024-25,False
59,Sat 1/11,@BAL,L28-14,20,29,270,69.0,9.3,2,0,...,0.0,5.0,2024,Russell Wilson,14881,Wild Card,russell wilson,2025-01-11,2024-25,True
60,Mon 1/13,@LAR,L27-9,25,40,245,62.5,6.1,1,1,...,0.0,17.0,2024,Sam Darnold,3912547,Wild Card,sam darnold,2025-01-13,2024-25,True
61,Sun 1/11,vsBUF,L27-24,18,30,207,60.0,6.9,3,2,...,0.0,18.0,2025,Trevor Lawrence,4360310,Wild Card,trevor lawrence,2026-01-11,2025-26,True


In [14]:
df_props

,player,away_team,home_team,game_time,market,prop_line,over_odds,under_odds,bookmaker,bookmaker_last_update,market_last_update,fetch_date,season,season_str,player_normalized,game_date
0,C.J. Stroud,Cleveland Browns,Houston Texans,2024-01-13T21:30:00Z,player_rush_yds,11.5,-110,-110.0,fanduel,2024-01-13T16:53:34Z,2024-01-13T16:56:24Z,2024-01-13,2023-24,2023-24,cj stroud,2024-01-13
1,Devin Singletary,Cleveland Browns,Houston Texans,2024-01-13T21:30:00Z,player_rush_yds,66.5,-110,-110.0,fanduel,2024-01-13T16:53:34Z,2024-01-13T16:56:24Z,2024-01-13,2023-24,2023-24,devin singletary,2024-01-13
2,Jerome Ford,Cleveland Browns,Houston Texans,2024-01-13T21:30:00Z,player_rush_yds,42.5,-110,-110.0,fanduel,2024-01-13T16:53:34Z,2024-01-13T16:56:24Z,2024-01-13,2023-24,2023-24,jerome ford,2024-01-13
3,Kareem Hunt,Cleveland Browns,Houston Texans,2024-01-13T21:30:00Z,player_rush_yds,24.5,-110,-110.0,fanduel,2024-01-13T16:53:34Z,2024-01-13T16:56:24Z,2024-01-13,2023-24,2023-24,kareem hunt,2024-01-13
4,C.J. Stroud,Cleveland Browns,Houston Texans,2024-01-13T21:30:00Z,player_rush_yds,11.5,-110,-120.0,draftkings,2024-01-13T16:54:51Z,2024-01-13T16:52:19Z,2024-01-13,2023-24,2023-24,cj stroud,2024-01-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1374,Aaron Rodgers,Houston Texans,Pittsburgh Steelers,2026-01-13T01:00:00Z,player_rush_yds,0.5,-130,100.0,bovada,2026-01-12T16:55:19Z,2026-01-12T16:55:55Z,2026-01-12,2025-26,2025-26,aaron rodgers,2026-01-13
1375,C.J. Stroud,Houston Texans,Pittsburgh Steelers,2026-01-13T01:00:00Z,player_rush_yds,12.5,-110,-120.0,bovada,2026-01-12T16:55:19Z,2026-01-12T16:55:55Z,2026-01-12,2025-26,2025-26,cj stroud,2026-01-13
1376,Jaylen Warren,Houston Texans,Pittsburgh Steelers,2026-01-13T01:00:00Z,player_rush_yds,52.5,-115,-115.0,bovada,2026-01-12T16:55:19Z,2026-01-12T16:55:55Z,2026-01-12,2025-26,2025-26,jaylen warren,2026-01-13
1377,Kenneth Gainwell,Houston Texans,Pittsburgh Steelers,2026-01-13T01:00:00Z,player_rush_yds,29.5,105,-135.0,bovada,2026-01-12T16:55:19Z,2026-01-12T16:55:55Z,2026-01-12,2025-26,2025-26,kenneth gainwell,2026-01-13


In [ ]:
# df_joined

,date,opponent,result,completions,attempts,passing_yards,passing_cmp%,passing_avg,passing_tds,interceptions,...,market,prop_line,over_odds,under_odds,bookmaker,bookmaker_last_update,market_last_update,fetch_date,season_prop,player_normalized
0,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,player_rush_yds,7.5,-115,-105.0,fanduel,2024-01-21T16:54:15Z,2024-01-21T16:55:15Z,2024-01-21,2023-24,baker mayfield
1,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,player_rush_yds,8.5,-114,-114.0,betonlineag,2024-01-21T16:54:50Z,2024-01-21T16:54:24Z,2024-01-21,2023-24,baker mayfield
2,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,player_rush_yds,8.5,-117,-112.0,betrivers,2024-01-21T16:54:14Z,2024-01-21T16:55:03Z,2024-01-21,2023-24,baker mayfield
3,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,player_rush_yds,8.5,-117,-112.0,unibet_us,2024-01-21T16:54:14Z,2024-01-21T16:55:10Z,2024-01-21,2023-24,baker mayfield
4,Sun 1/21,@DET,L31-23,26,41,349,63.4,8.5,3,2,...,player_rush_yds,7.5,-125,-105.0,bovada,2024-01-21T16:54:14Z,2024-01-21T16:54:32Z,2024-01-21,2023-24,baker mayfield
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,Sun 1/11,vsBUF,L27-24,18,30,207,60.0,6.9,3,2,...,player_rush_yds,25.5,-113,-118.0,betrivers,2026-01-11T16:55:16Z,2026-01-11T16:55:20Z,2026-01-11,2025-26,trevor lawrence
290,Sun 1/11,vsBUF,L27-24,18,30,207,60.0,6.9,3,2,...,player_rush_yds,27.5,-114,-114.0,betonlineag,2026-01-11T16:55:16Z,2026-01-11T16:54:22Z,2026-01-11,2025-26,trevor lawrence
291,Sun 1/11,vsBUF,L27-24,18,30,207,60.0,6.9,3,2,...,player_rush_yds,26.5,-115,-115.0,williamhill_us,2026-01-11T16:53:39Z,2026-01-11T16:55:52Z,2026-01-11,2025-26,trevor lawrence
292,Sun 1/11,vsBUF,L27-24,18,30,207,60.0,6.9,3,2,...,player_rush_yds,27.5,100,-135.0,betmgm,2026-01-11T16:55:04Z,2026-01-11T16:54:57Z,2026-01-11,2025-26,trevor lawrence


In [16]:
df_consensus

,athlete,athlete_id,game_date,season,season_str,opponent,playoff_round,rushing_yds,rushing_car,rushing_avg,passing_yards,passing_tds,attempts,result,is_first_playoff_game,consensus_line,num_bookmakers,diff_vs_consensus,beat_line
0,Baker Mayfield,3052587,2024-01-21,2023,2023-24,@DET,Divisional,15.0,2.0,7.5,349,3,41,L31-23,False,8.5,9,6.5,True
1,Bo Nix,4426338,2025-01-12,2024,2024-25,@BUF,Wild Card,43.0,4.0,10.8,144,1,22,L31-7,True,28.5,7,14.5,True
2,Brock Purdy,4361741,2024-01-28,2023,2023-24,vsDET,Conference Championship,48.0,5.0,9.6,267,1,31,W34-31,False,7.5,8,40.5,True
3,Brock Purdy,4361741,2024-02-11,2023,2023-24,vsKC,Super Bowl,12.0,3.0,4.0,255,1,38,L25-22 OT,False,13.0,8,-1.0,False
4,Brock Purdy,4361741,2026-01-11,2025,2025-26,@PHI,Wild Card,24.0,9.0,2.7,262,2,31,W23-19,False,16.5,7,7.5,True
5,Bryce Young,4685720,2026-01-10,2025,2025-26,vsLAR,Wild Card,24.0,3.0,8.0,264,1,40,L34-31,True,16.5,7,7.5,True
6,C.J. Stroud,4432577,2024-01-13,2023,2023-24,vsCLE,Wild Card,1.0,1.0,1.0,274,3,21,W45-14,True,12.0,8,-11.0,False
7,C.J. Stroud,4432577,2024-01-20,2023,2023-24,@BAL,Divisional,9.0,3.0,3.0,175,0,33,L34-10,False,8.5,8,0.5,True
8,C.J. Stroud,4432577,2025-01-11,2024,2024-25,vsLAC,Wild Card,42.0,6.0,7.0,282,1,33,W32-12,False,13.5,7,28.5,True
9,C.J. Stroud,4432577,2025-01-18,2024,2024-25,@KC,Divisional,42.0,6.0,7.0,245,0,28,L23-14,False,15.5,7,26.5,True


In [30]:
df_grouped = (
    df_consensus.groupby('is_first_playoff_game')
    .agg({
        'beat_line': ['count', 'sum', 'mean'],
        'diff_vs_consensus': 'mean'
    })
    .round(2)
)

# Flatten columns
df_grouped.columns = ['Total Games', 'Wins', 'Went Over Line', 'Avg Margin']
df_grouped = df_grouped.reset_index()

# Clean up the index column
df_grouped['Status'] = df_grouped['is_first_playoff_game'].map({
    True: '🆕 First Playoff Game',
    False: '🏆 Veteran QB'
})

df_grouped = df_grouped[['Status', 'Total Games', 'Wins', 'Went Over Line', 'Avg Margin']]

df_grouped

,Status,Total Games,Wins,Went Over Line,Avg Margin
0,🏆 Veteran QB,31,15,0.48,3.82
1,🆕 First Playoff Game,11,8,0.73,11.14
